# Automated Technical File demo

This notebook shows how a raw JSONL robot-run ledger becomes a Markdown corpus page for downstream Mistral retrieval and Q&A.

**Cells 1-2:** Ledger to Markdown conversion (CB-04/CB-05)
**Cells 3-4:** Document Library + Cited Q&A (CB-06 - Misty)
**Cell 5:** Structured-outputs log analysis (CB-08 - Misty)
**Cells 6-7:** Voice Interface + Voxtral Voice Loop (CB-09)


In [ ]:
from pathlib import Path
import subprocess
import sys

root = Path.cwd()
if not (root / "tools" / "ledger_to_md.py").exists():
    root = root / "third_party" / "automated-technical-file"

ledger = root / "artifacts" / "ledger" / "sample_events.jsonl"
wiki_page = root / "artifacts" / "wiki" / "sample_run_summary.md"

subprocess.run(
    [
        sys.executable,
        str(root / "tools" / "ledger_to_md.py"),
        "--input",
        str(ledger),
        "--output",
        str(wiki_page),
        "--title",
        "Sample Automated Technical File Run",
    ],
    check=True,
)

print(wiki_page.read_text(encoding="utf-8")[:1200])

In [ ]:
# Cell 3: Build Mistral Document Library over wiki corpus MD files

import os
from pathlib import Path
from typing import Optional

# Configuration
MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY')
BACKEND = os.environ.get('BACKEND', 'auto').lower()

# Model IDs from CB-01 SPEC_models.md
MODEL_HOSTED = 'mistral-large-latest'
MODEL_LOCAL = 'ministral-3b-latest'

# Wiki corpus file paths (relative to automated-technical-file dir)
WIKI_CORPUS_FILES = [
    'notebooks/Overview.md',
    'notebooks/Topics/Compliance.md',
    'notebooks/Topics/BiddingRules.md',
    'notebooks/Subsystems/HardwareInterface.md'
]

try:
    from mistralai import Mistral
except ImportError:
    print('ERROR: mistralai package not installed. Run: pip install mistralai')
    LIBRARY_ID = None
    DOCUMENT_IDS = []
else:
    def create_document_library(api_key, library_name='RobotRoss ATF Wiki'):
        client = Mistral(api_key=api_key)
        library = client.beta.libraries.create(
            name=library_name,
            description='Wiki corpus for RobotRoss Automated Technical File'
        )
        print(f'Created library: {library.name} (ID: {library.id})')
        uploaded_docs = []
        base_path = Path.cwd() / 'third_party' / 'automated-technical-file'
        for file_path in WIKI_CORPUS_FILES:
            full_path = base_path / file_path
            if not full_path.exists():
                print(f'WARNING: File not found: {full_path}')
                continue
            with open(full_path, 'rb') as f:
                doc = client.beta.libraries.documents.upload(
                    library_id=library.id,
                    file={'file_name': file_path, 'content': f.read()}
                )
            uploaded_docs.append(doc)
            print(f'Uploaded: {file_path} (doc ID: {doc.id})')
        return {'library': library, 'documents': uploaded_docs}
    
    if BACKEND == 'hosted' and MISTRAL_API_KEY:
        library_data = create_document_library(MISTRAL_API_KEY)
        LIBRARY_ID = library_data['library'].id
        DOCUMENT_IDS = [d.id for d in library_data['documents']]
        print(f'Library created with {len(DOCUMENT_IDS)} documents')
    elif BACKEND == 'local':
        print('Local backend: Document Library uses in-notebook index')
        LIBRARY_ID = None
        DOCUMENT_IDS = []
    else:
        print('Auto backend: Try hosted first, fall back to local')
        LIBRARY_ID = None
        DOCUMENT_IDS = []

In [ ]:
# Cell 4: Cited Q&A using Agents API + native Citations

def create_cited_qa_agent(api_key, library_id=None):
    client = Mistral(api_key=api_key)
    tools = []
    if library_id:
        tools.append({'type': 'document_library', 'document_library': {'library_ids': [library_id]}})
    agent = client.agents.create(
        name='ATF Q&A Assistant',
        model=MODEL_HOSTED,
        description='Answer questions about RobotRoss ATF with cited sources',
        instructions='You are a helpful assistant. Always ground answers in documents. Include citations.',
        tools=tools
    )
    return agent

def ask_with_citations(agent, question):
    client = Mistral(api_key=MISTRAL_API_KEY)
    response = client.agents.completions.create(
        agent_id=agent.id,
        messages=[{'role': 'user', 'content': question}]
    )
    answer_parts = []
    sources = []
    for chunk in response.choices[0].message.content:
        if hasattr(chunk, 'type'):
            if chunk.type == 'text':
                answer_parts.append(chunk.text)
            elif chunk.type == 'tool_reference':
                sources.append(chunk.tool_reference.reference)
    answer = ''.join(answer_parts)
    if sources:
        answer += '\n\n---\n**Sources:**\n' + '\n'.join(f'- {s}' for s in sources)
    return answer

print('=' * 60)
print('Example: Cited Q&A with Mistral Document Library')
print('=' * 60)

if BACKEND == 'hosted' and MISTRAL_API_KEY and LIBRARY_ID:
    agent = create_cited_qa_agent(MISTRAL_API_KEY, LIBRARY_ID)
    question = "What is the bidding rule for the Wall of Fame?"
    answer = ask_with_citations(agent, question)
    print(f'Q: {question}')
    print(f'A: {answer}')
elif BACKEND == 'local':
    print('Local backend: Use atf_qa.py from ATF/tools/')
    print('Example: python3 ATF/tools/atf_qa.py "What is the bidding rule?"')
else:
    print('Set BACKEND=hosted and MISTRAL_API_KEY for Document Library')
    print('Or BACKEND=local for atf_qa.py corpus approach')

In [ ]:
# Cell 5: Structured-outputs log analysis# Take a slice of sample_events.jsonl, use Mistral Structured Outputs (JSON schema)# to extract typed analysis: patterns, anomalies, metrics, recommendations.# Model IDs from CB-01 SPEC_models.mdimport osimport jsonfrom pathlib import Pathfrom pydantic import BaseModelfrom mistralai import Mistral# ============================================================# Structured Output Schema# ============================================================class LogSliceAnalysis(BaseModel):    """Typed analysis of a slice of robot-run ledger events."""    patterns: list[str]  # Recurring themes or sequences in the log    anomalies: list[str]  # Unexpected or outlier events    metrics: dict  # Quantitative summary (counts, durations, etc.)    recommendations: list[str]  # Actionable insights from the analysis# ============================================================# Configuration# ============================================================MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY')BACKEND = os.environ.get('BACKEND', 'auto').lower()MODEL = 'mistral-large-latest'  # From CB-01 SPEC_models.mdLEDGER_PATH = Path('artifacts/ledger/sample_events.jsonl')# ============================================================# Load and slice events# ============================================================def load_events(filepath, limit=None):    """Load JSONL events, optionally limiting to a slice."""    events = []    with open(filepath) as f:        for line in f:            events.append(json.loads(line))            if limit and len(events) >= limit:                break    return events# Take a meaningful slice: AI events + job start/end for contextall_events = load_events(LEDGER_PATH)ai_events = [e for e in all_events if e.get('event_category') == 'AI']job_events = [e for e in all_events if e.get('event_category') == 'JOB']system_events = [e for e in all_events if e.get('event_category') == 'SYSTEM']# Use AI + Job events for analysis sliceslice_events = ai_events + job_eventsslice_jsonl = '\n'.join(json.dumps(e) for e in slice_events)# ============================================================# Structured Output Analysis# ============================================================def analyze_with_structured_outputs(api_key, events_jsonl, model=MODEL):    """Use Mistral Structured Outputs to analyze event slice."""    client = Mistral(api_key=api_key)        prompt = f"""Analyze the following robot-run ledger event slice.Extract patterns, anomalies, metrics, and recommendations.Be concise and technical.Events:{events_jsonl}Provide your analysis as valid JSON matching the schema."""        response = client.chat.parse(        model=model,        messages=[{'role': 'user', 'content': prompt}],        response_format=LogSliceAnalysis    )        return response# ============================================================# Run analysis# ============================================================if MISTRAL_API_KEY:    analysis = analyze_with_structured_outputs(MISTRAL_API_KEY, slice_jsonl)    print('=== Structured Output Analysis ===')    print(json.dumps(analysis.model_dump(), indent=2))else:    print('MISTRAL_API_KEY not found. Returning stub analysis for testing.')    print('To run with real analysis: set MISTRAL_API_KEY env var')    # Stub response matching the schema    stub = LogSliceAnalysis(        patterns=[            'AI sketch generation followed by job execution',            'Token usage tracked for each AI request'        ],        anomalies=[            'No anomalies detected in sample slice'        ],        metrics={'ai_requests': 1, 'jobs': 1, 'total_duration': 300},        recommendations=[            'Consider batching AI requests for efficiency',            'Add more detailed timing metrics'        ]    )    print(json.dumps(stub.model_dump(), indent=2))

## Step 3: Voice Interface (Optional / Stub-able)

This section demonstrates the Voxtral Voice pipeline: **Voxtral Transcribe 2** for speech-to-text, **Mistral** for reasoning, and **Voxtral TTS** for spoken responses. 

This cell is marked as stub-able so it can be skipped if a microphone or specific audio dependencies are not available in your environment.

In [ ]:
# Cell 6: Voxtral Voice Loop (STT -> LLM -> TTS)
import os
import sys
from pathlib import Path

# Model IDs from CB-01 SPEC_models.md
STT_MODEL = "voxtral-mini-2602"
TTS_MODEL = "voxtral-mini-tts-2603"
REASONING_MODEL = "mistral-large-latest"

try:
    # Add tools/voice to path
    voice_tools_path = Path.cwd() / "third_party" / "automated-technical-file" / "tools" / "voice"
    if not voice_tools_path.exists():
        voice_tools_path = Path.cwd() / "tools" / "voice"
    sys.path.append(str(voice_tools_path))

    from listen import VoxtralListener
    from speak import speak
    from mistralai import Mistral

    def voice_interaction_loop():
        """Runs a short voice interaction demo."""
        api_key = os.environ.get("MISTRAL_API_KEY")
        if not api_key:
            print("MISTRAL_API_KEY not found. Skipping voice interaction.")
            return

        client = Mistral(api_key=api_key)
        listener = VoxtralListener(model_id=STT_MODEL)

        print("--- Voice Interaction Started (Robot Ross Mode) ---")
        print("Speak now (ask about the Automated Technical File)...")
        
        user_text = listener.listen()
        if not user_text:
            print("No speech detected.")
            return

        # Reasoning Step
        response = client.chat.complete(
            model=REASONING_MODEL,
            messages=[{"role": "user", "content": user_text}]
        )
        ai_text = response.choices[0].message.content
        print(f"AI: {ai_text}")

        # Speech Step
        speak(ai_text)

    # Uncomment to run (requires microphone and MISTRAL_API_KEY)
    # voice_interaction_loop()
    print("Voice interaction cell ready. (Uncomment the call to run manually)")

except ImportError as e:
    print(f"Voice dependencies not met: {e}")
    print("Ensure 'sounddevice', 'soundfile', and 'mistralai' are installed.")
except Exception as e:
    print(f"Voice cell stubbed: {e}")